INSTALL LIBRARY

In [14]:
!pip install transformers datasets scikit-learn sastrawi

IMPORT

In [15]:
import pandas as pd
import numpy as np
import re
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

LOAD DATASET

In [16]:
df = pd.read_csv("/content/ulasan_playstore_livin.csv", sep=';', on_bad_lines='skip')

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df[['content']].dropna()

print(df.head())

                                             content
0  apalah livin tidak bisa login, tombol login ga...
1  Saya senang ada aplikasi ini karena sangat mem...
2                                  sering update trs
3  SERING GANGGUAN WKWKW ada maintenance sampe ti...
4  aplikasi tidak bisa digunakan, PD saat daftar ...


PREPROCESSING

In [18]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean'] = df['content'].apply(preprocess)
df.head()

,content,clean
0,"apalah livin tidak bisa login, tombol login ga...",apalah livin tidak bisa login tombol login gak...
1,Saya senang ada aplikasi ini karena sangat mem...,saya senang ada aplikasi ini karena sangat mem...
2,sering update trs,sering update trs
3,SERING GANGGUAN WKWKW ada maintenance sampe ti...,sering gangguan wkwkw ada maintenance sampe ti...
4,"aplikasi tidak bisa digunakan, PD saat daftar ...",aplikasi tidak bisa digunakan pd saat daftar k...


AUTO LABEL EMOSI (NLP + RULE HYBRID 🔥)

In [39]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

def auto_emotion_label(text):
    text_low = text.lower()

    result = sentiment_model(text[:512])[0]['label']

    # DEBUG (boleh aktifkan sementara)
    # print(result)

    if result == "positive":
        return "senang"

    elif result == "negative":
        if any(k in text_low for k in ["marah", "kesal", "parah"]):
            return "marah"
        elif any(k in text_low for k in ["sedih", "mengecewakan"]):
            return "sedih"
        elif any(k in text_low for k in ["error", "gagal", "lambat", "tidak bisa"]):
            return "kecewa"
        else:
            return "kecewa"

    return "netral"

df['emotion'] = df['clean'].apply(auto_emotion_label)

print(df['emotion'].value_counts())

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

emotion
kecewa    513
senang    308
netral    158
marah      17
sedih       3
Name: count, dtype: int64


DETEKSI SARKASEMSE

In [40]:
def detect_sarcasm(text):
    text = text.lower()

    patterns = [
        ("bagus", "error"),
        ("mantap", "lambat"),
        ("keren", "gagal"),
        ("hebat", "buruk")
    ]

    for pos, neg in patterns:
        if pos in text and neg in text:
            return 1

    return 0

df['sarcasm'] = df['clean'].apply(detect_sarcasm)
df.head()

,content,clean,emotion,sarcasm,label_id
0,"apalah livin tidak bisa login, tombol login ga...",apalah livin tidak bisa login tombol login gak...,kecewa,0,4
1,Saya senang ada aplikasi ini karena sangat mem...,saya senang ada aplikasi ini karena sangat mem...,senang,0,4
2,sering update trs,sering update trs,senang,0,4
3,SERING GANGGUAN WKWKW ada maintenance sampe ti...,sering gangguan wkwkw ada maintenance sampe ti...,kecewa,0,4
4,"aplikasi tidak bisa digunakan, PD saat daftar ...",aplikasi tidak bisa digunakan pd saat daftar k...,netral,0,4


ENCODING LABEL

In [53]:
label2id = {
    "senang": 0,
    "marah": 1,
    "sedih": 2,
    "kecewa": 3,
    "netral": 4
}

id2label = {v: k for k, v in label2id.items()}

df['label_id'] = df['emotion'].map(label2id)
df.head(20)

,content,clean,emotion,sarcasm,label_id
0,"apalah livin tidak bisa login, tombol login ga...",apalah livin tidak bisa login tombol login gak...,kecewa,0,3
1,Saya senang ada aplikasi ini karena sangat mem...,saya senang ada aplikasi ini karena sangat mem...,senang,0,0
2,sering update trs,sering update trs,senang,0,0
3,SERING GANGGUAN WKWKW ada maintenance sampe ti...,sering gangguan wkwkw ada maintenance sampe ti...,kecewa,0,3
4,"aplikasi tidak bisa digunakan, PD saat daftar ...",aplikasi tidak bisa digunakan pd saat daftar k...,netral,0,4
5,"kok saldo saya tiba tiba berkurang, tidak ada ...",kok saldo saya tiba tiba berkurang tidak ada r...,kecewa,0,3
6,"Mohon bantuannya, transaksi uang masuk dan kel...",mohon bantuannya transaksi uang masuk dan kelu...,netral,0,4
7,"lama"" ganti bank kalau kaya gini ngurus di ban...",lama ganti bank kalau kaya gini ngurus di bank...,kecewa,0,3
8,"Perbaikan kok setiap hari, orang mau transaksi...",perbaikan kok setiap hari orang mau transaksi ...,kecewa,0,3
9,tiap malem pasti offline pemeliharaan,tiap malem pasti offline pemeliharaan,kecewa,0,3


SPLIT DATA

In [42]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clean'],
    df['label_id'],
    test_size=0.2,
    random_state=42,
    stratify=df['label_id']
)

TOKENIZATION

In [43]:
model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)

DATASET CLASS

In [44]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)

LOAD MODEL

In [45]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

TRAINING CONFIG

In [46]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5
    # evaluation_strategy="epoch" # Removed to fix TypeError
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Step,Training Loss
500,0.274912


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.27491162109375, metrics={'train_runtime': 117.0554, 'train_samples_per_second': 34.129, 'train_steps_per_second': 4.271, 'total_flos': 190932810664770.0, 'train_loss': 0.27491162109375, 'epoch': 5.0})

EVALUASI

In [47]:
predictions = trainer.predict(val_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = val_labels.values

print(classification_report(
    y_true,
    y_pred,
    target_names=list(label2id.keys()),
    labels=list(id2label.keys()) # Explicitly specify all possible labels
))

              precision    recall  f1-score   support

      senang       0.92      0.87      0.89        62
       marah       1.00      1.00      1.00         3
       sedih       0.00      0.00      0.00         0
      kecewa       0.90      0.92      0.91       103
      netral       0.67      0.69      0.68        32

    accuracy                           0.87       200
   macro avg       0.70      0.70      0.70       200
weighted avg       0.87      0.87      0.87       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

PREDICT FINAL

In [48]:
def predict(text):
    model.eval()

    clean_text = preprocess(text)

    inputs = tokenizer(clean_text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    pred_id = torch.argmax(probs, dim=1).item()

    emotion = id2label[pred_id]
    sarcasm = detect_sarcasm(text)

    return emotion, sarcasm

TEST

In [58]:
text = "Orang Karyawan nyaa pada ramah, sopan, baik.."

emotion, sarcasm = predict(text)

print("Teks:", text)
print("Emosi:", emotion)
print("Sarkasme:", "Ya" if sarcasm else "Tidak")

Teks: Orang Karyawan nyaa pada ramah, sopan, baik..
Emosi: senang
Sarkasme: Tidak


BULK TESTING

In [61]:
def predict_bulk(file_path, output_path="hasil_prediksi.csv"):
    df_test = pd.read_csv(file_path)

    if 'content' not in df_test.columns:
        raise ValueError("Kolom 'content' wajib ada")

    emotions = []
    sarcasms = []

    for text in df_test['content']:
        emotion, sarcasm = predict(text)

        emotions.append(emotion)
        sarcasms.append("Ya" if sarcasm else "Tidak")

    df_test['emotion'] = emotions
    df_test['sarcasm'] = sarcasms

    df_test.to_csv(output_path, index=False)

    print("✅ Hasil disimpan di:", output_path)

    return df_test

# Create a dummy test_bulk.csv file for demonstration
df.head(10)[['content']].to_csv('test_bulk.csv', index=False)

result = predict_bulk("test_bulk.csv")
result.head(25)

✅ Hasil disimpan di: hasil_prediksi.csv


,content,emotion,sarcasm
0,"apalah livin tidak bisa login, tombol login ga...",kecewa,Tidak
1,Saya senang ada aplikasi ini karena sangat mem...,senang,Tidak
2,sering update trs,kecewa,Tidak
3,SERING GANGGUAN WKWKW ada maintenance sampe ti...,kecewa,Tidak
4,"aplikasi tidak bisa digunakan, PD saat daftar ...",netral,Tidak
5,"kok saldo saya tiba tiba berkurang, tidak ada ...",kecewa,Tidak
6,"Mohon bantuannya, transaksi uang masuk dan kel...",netral,Tidak
7,"lama"" ganti bank kalau kaya gini ngurus di ban...",kecewa,Tidak
8,"Perbaikan kok setiap hari, orang mau transaksi...",kecewa,Tidak
9,tiap malem pasti offline pemeliharaan,kecewa,Tidak
